In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
from neuralforecast.core import NeuralForecast
from neuralforecast.models import NBEATS, NHITS
from neuralforecast.auto import AutoMLP
from neuralforecast.utils import AirPassengersDF

In [ ]:
# We define the train df. 
Y_df = AirPassengersDF
Y_df.head()

In [ ]:
Y_train_df = Y_df[Y_df.ds<='1959-12-31'] # 132 train
Y_test_df = Y_df[Y_df.ds>'1959-12-31'] # 12 test

In [ ]:
%%capture
horizon = len(Y_test_df)
config = dict(max_steps=1, input_size=12)
models = [NBEATS(input_size=2 * horizon, h=horizon, max_epochs=1),
          NBEATS(input_size=3 * horizon, h=horizon, max_epochs=1),
          NHITS(input_size=2 * horizon, h=horizon, max_epochs=1),
          AutoMLP(h=12, config=config, num_samples=1, cpus=1)]
fcst = NeuralForecast(models=models, freq='M')

In [ ]:
fcst.fit(df=Y_train_df)

In [ ]:
Y_hat_df = fcst.predict().reset_index()
fig, ax = plt.subplots(1, 1, figsize = (20, 7))
Y_hat_df = Y_test_df.merge(Y_hat_df, how='left', on=['unique_id', 'ds'])
plot_df = pd.concat([Y_train_df, Y_hat_df]).set_index('ds')

plot_df[['y', 'NBEATS', 'NBEATS1', 'NHITS', 'AutoMLP']].plot(ax=ax, linewidth=2)

ax.set_title('AirPassengers Forecast', fontsize=22)
ax.set_ylabel('Monthly Passengers', fontsize=20)
ax.set_xlabel('Timestamp [t]', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()

In [ ]:
fcst.save(path='./checkpoints/test_run/', model_index=None, overwrite=True, save_dataset=True)

In [ ]:
fcst2 = NeuralForecast.load(path='./checkpoints/test_run/')

In [ ]:
Y_hat_df = fcst2.predict().reset_index()
fig, ax = plt.subplots(1, 1, figsize = (20, 7))
Y_hat_df = Y_test_df.merge(Y_hat_df, how='left', on=['unique_id', 'ds'])
plot_df = pd.concat([Y_train_df, Y_hat_df]).set_index('ds')

plot_df[['y', 'NBEATS', 'NBEATS1', 'NHITS', 'MLP']].plot(ax=ax, linewidth=2)

ax.set_title('AirPassengers Forecast', fontsize=22)
ax.set_ylabel('Monthly Passengers', fontsize=20)
ax.set_xlabel('Timestamp [t]', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()